# Análise de Dados de Produtos - Marketplace Search System

Este notebook permite visualizar e analisar os dados de produtos do sistema de busca do marketplace. 
Utilizamos pandas, matplotlib e seaborn para criar visualizações interativas e insights sobre os produtos.

## Estrutura do Notebook:
1. **Importação de Bibliotecas** - Pandas, Matplotlib, Seaborn, Requests
2. **Carregamento de Dados** - Conectar ao Elasticsearch e carregar produtos
3. **Exploração e Limpeza** - Análise exploratória dos dados
4. **Visualizações Básicas** - Gráficos de distribuição e categorias
5. **Analytics Avançados** - Correlações e tendências
6. **Filtros Interativos** - Exploração dinâmica dos dados

## 1. Importação de Bibliotecas

Importamos todas as bibliotecas necessárias para análise de dados e visualização.

In [ ]:
# Bibliotecas para manipulação de dados
import pandas as pd
import numpy as np
import json
import requests
from datetime import datetime, timedelta

# Bibliotecas para visualização
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Configurações do pandas e matplotlib
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")

# Configurações do Elasticsearch
ELASTICSEARCH_URL = "http://localhost:9200"
INDEX_NAME = "products"

print("✅ Bibliotecas importadas com sucesso!")
print(f"📊 Pandas version: {pd.__version__}")
print(f"📈 Matplotlib version: {plt.matplotlib.__version__}")
print(f"🎨 Seaborn version: {sns.__version__}")

✅ Bibliotecas importadas com sucesso!
📊 Pandas version: 2.3.3
📈 Matplotlib version: 3.10.7
🎨 Seaborn version: 0.13.2


## 2. Carregamento de Dados do Elasticsearch

Conectamos ao Elasticsearch e carregamos os dados de produtos para análise.

In [2]:
def load_products_from_elasticsearch(es_url=ELASTICSEARCH_URL, index=INDEX_NAME, size=1000):
    """
    Carrega produtos do Elasticsearch e converte para DataFrame do pandas
    Estrutura atualizada para usar campos no nível raiz.
    """
    try:
        # Verificar se o Elasticsearch está rodando
        health_url = f"{es_url}/_cluster/health"
        health_response = requests.get(health_url)
        if health_response.status_code != 200:
            print("❌ Elasticsearch não está rodando. Verifique se o container está ativo.")
            return None
        
        print("✅ Elasticsearch está rodando!")
        
        # Buscar todos os produtos
        search_url = f"{es_url}/{index}/_search"
        query = {
            "query": {"match_all": {}},
            "size": size
        }
        
        response = requests.post(search_url, json=query, headers={'Content-Type': 'application/json'})
        
        if response.status_code != 200:
            print(f"❌ Erro ao buscar produtos: {response.text}")
            return None
        
        data = response.json()
        hits = data.get('hits', {}).get('hits', [])
        
        if not hits:
            print("⚠️  Nenhum produto encontrado no Elasticsearch.")
            return None
        
        # Converter para DataFrame usando a estrutura corrigida
        products = []
        for hit in hits:
            source = hit['_source']
            product = {
                'id': source.get('id'),
                'title': source.get('title'),  # Usando 'title' em vez de 'name'
                'description': source.get('description'),
                'price': source.get('price'),
                'currency': source.get('currency', 'BRL'),
                
                # Campos do nível raiz (corrigidos)
                'category_id': source.get('category_id'),
                'category_name': source.get('category_name'),
                'brand_id': source.get('brand_id'),
                'brand_name': source.get('brand_name'),
                'seller_id': source.get('seller_id'),
                'seller_name': source.get('seller_name'),
                'seller_reputation': source.get('seller_reputation'),
                'stock_quantity': source.get('stock_quantity'),
                
                # Campos de métricas diretos
                'views': source.get('views'),
                'sales': source.get('sales'),
                'rating': source.get('rating'),
                
                # Campos de data
                'created_at': source.get('created_at'),
                'updated_at': source.get('updated_at'),
                
                # Status e outros campos
                'status': source.get('status', {}).get('value', 'UNKNOWN'),
                'popularity_score': source.get('popularity_score'),
                'price_range': source.get('price_range'),
                
                # Tags (se existirem)
                'tags': ', '.join(source.get('tags', [])) if source.get('tags') else ''
            }
            products.append(product)
        
        df = pd.DataFrame(products)
        
        # Conversões de tipo para campos numéricos
        numeric_fields = ['price', 'stock_quantity', 'seller_reputation', 'views', 'sales', 'rating', 'popularity_score']
        for field in numeric_fields:
            if field in df.columns:
                df[field] = pd.to_numeric(df[field], errors='coerce')
        
        # Conversões de data
        date_columns = ['created_at', 'updated_at']
        for col in date_columns:
            if col in df.columns:
                df[col] = pd.to_datetime(df[col], errors='coerce')
        
        print(f"✅ Carregados {len(df)} produtos do Elasticsearch!")
        print(f"📊 Campos disponíveis: {list(df.columns)}")
        return df
        
    except Exception as e:
        print(f"❌ Erro ao conectar com Elasticsearch: {str(e)}")
        return None

# Carregar os dados com a estrutura corrigida
print("🔄 Recarregando dados do Elasticsearch com estrutura corrigida...")
df_products = load_products_from_elasticsearch()

if df_products is not None:
    print(f"\n📊 Dataset carregado com sucesso:")
    print(f"   • {len(df_products)} produtos")
    print(f"   • {len(df_products.columns)} colunas")
    
    # Verificar se os campos problemáticos anteriores agora têm dados
    problematic_fields = ['seller_reputation', 'stock_quantity', 'created_at', 'updated_at']
    print(f"\n🔍 Verificação dos campos corrigidos:")
    for field in problematic_fields:
        if field in df_products.columns:
            valid_count = df_products[field].notna().sum()
            total = len(df_products)
            percentage = (valid_count / total) * 100
            print(f"   ✅ {field}: {valid_count}/{total} ({percentage:.1f}%) valores válidos")
        else:
            print(f"   ❌ {field}: Campo não encontrado")
    
    if 'created_at' in df_products.columns and df_products['created_at'].notna().any():
        print(f"   📅 Período: {df_products['created_at'].min()} até {df_products['created_at'].max()}")
else:
    print("⚠️  Erro ao carregar dados do Elasticsearch. Verifique se o serviço está rodando e se há dados indexados.")

🔄 Recarregando dados do Elasticsearch com estrutura corrigida...
✅ Elasticsearch está rodando!
✅ Carregados 100 produtos do Elasticsearch!
📊 Campos disponíveis: ['id', 'title', 'description', 'price', 'currency', 'category_id', 'category_name', 'brand_id', 'brand_name', 'seller_id', 'seller_name', 'seller_reputation', 'stock_quantity', 'views', 'sales', 'rating', 'created_at', 'updated_at', 'status', 'popularity_score', 'price_range', 'tags']

📊 Dataset carregado com sucesso:
   • 100 produtos
   • 22 colunas

🔍 Verificação dos campos corrigidos:
   ✅ seller_reputation: 100/100 (100.0%) valores válidos
   ✅ stock_quantity: 100/100 (100.0%) valores válidos
   ✅ created_at: 100/100 (100.0%) valores válidos
   ✅ updated_at: 100/100 (100.0%) valores válidos
   📅 Período: 2023-10-15 02:43:07.509906+00:00 até 2025-09-25 21:22:21.140004+00:00


## 3. Exploração e Limpeza dos Dados

Agora vamos explorar a estrutura dos dados e identificar padrões.

## 2.1. Diagnóstico de Qualidade dos Dados

Vamos verificar a integridade e completude dos dados carregados antes da análise.

In [3]:
# Diagnóstico completo da qualidade dos dados
print("=" * 60)
print("RELATÓRIO DE QUALIDADE DOS DADOS")
print("=" * 60)

# 1. Informações gerais
print(f"\n📊 INFORMAÇÕES GERAIS:")
print(f"   • Total de produtos: {len(df_products):,}")
print(f"   • Número de colunas: {len(df_products.columns)}")
print(f"   • Memória utilizada: {df_products.memory_usage().sum() / 1024 / 1024:.2f} MB")

# 2. Valores ausentes
print(f"\n❌ VALORES AUSENTES:")
missing_values = df_products.isnull().sum()
missing_percent = (missing_values / len(df_products)) * 100

for column in df_products.columns:
    missing_count = missing_values[column]
    missing_pct = missing_percent[column]
    if missing_count > 0:
        print(f"   • {column}: {missing_count:,} ({missing_pct:.1f}%)")
    else:
        print(f"   ✅ {column}: 0 (0.0%)")

# 3. Tipos de dados
print(f"\n📋 TIPOS DE DADOS:")
for column in df_products.columns:
    dtype = df_products[column].dtype
    print(f"   • {column}: {dtype}")

# 4. Estatísticas básicas para campos numéricos
print(f"\n📈 ESTATÍSTICAS NUMÉRICAS:")
numeric_columns = df_products.select_dtypes(include=['int64', 'float64']).columns
if len(numeric_columns) > 0:
    stats = df_products[numeric_columns].describe()
    print(stats)
else:
    print("   ⚠️ Nenhuma coluna numérica encontrada")

# 5. Verificação de campos categóricos
print(f"\n🏷️ CAMPOS CATEGÓRICOS:")
categorical_columns = df_products.select_dtypes(include=['object']).columns
for column in categorical_columns:
    unique_count = df_products[column].nunique()
    print(f"   • {column}: {unique_count} valores únicos")
    if unique_count <= 10:
        print(f"     Valores: {df_products[column].unique().tolist()}")

# 6. Verificação específica de campos problemáticos anteriores
print(f"\n🔍 VERIFICAÇÃO DE CAMPOS PROBLEMÁTICOS:")
problematic_fields = ['seller_reputation', 'stock_quantity', 'created_at', 'updated_at']

for field in problematic_fields:
    if field in df_products.columns:
        non_null_count = df_products[field].notna().sum()
        total_count = len(df_products)
        print(f"   • {field}: {non_null_count}/{total_count} valores válidos ({(non_null_count/total_count)*100:.1f}%)")
        
        # Para campos numéricos, mostrar alguns valores de exemplo
        if field in ['seller_reputation', 'stock_quantity']:
            sample_values = df_products[field].dropna().head(5).tolist()
            print(f"     Exemplos: {sample_values}")
            
        # Para campos de data, verificar formato
        if field in ['created_at', 'updated_at']:
            sample_values = df_products[field].dropna().head(3).tolist()
            print(f"     Exemplos: {sample_values}")
    else:
        print(f"   ❌ {field}: Campo não encontrado no dataset")

print("\n" + "=" * 60)

RELATÓRIO DE QUALIDADE DOS DADOS

📊 INFORMAÇÕES GERAIS:
   • Total de produtos: 100
   • Número de colunas: 22
   • Memória utilizada: 0.02 MB

❌ VALORES AUSENTES:
   ✅ id: 0 (0.0%)
   ✅ title: 0 (0.0%)
   ✅ description: 0 (0.0%)
   ✅ price: 0 (0.0%)
   ✅ currency: 0 (0.0%)
   ✅ category_id: 0 (0.0%)
   ✅ category_name: 0 (0.0%)
   ✅ brand_id: 0 (0.0%)
   ✅ brand_name: 0 (0.0%)
   ✅ seller_id: 0 (0.0%)
   ✅ seller_name: 0 (0.0%)
   ✅ seller_reputation: 0 (0.0%)
   ✅ stock_quantity: 0 (0.0%)
   ✅ views: 0 (0.0%)
   ✅ sales: 0 (0.0%)
   ✅ rating: 0 (0.0%)
   ✅ created_at: 0 (0.0%)
   ✅ updated_at: 0 (0.0%)
   ✅ status: 0 (0.0%)
   ✅ popularity_score: 0 (0.0%)
   ✅ price_range: 0 (0.0%)
   ✅ tags: 0 (0.0%)

📋 TIPOS DE DADOS:
   • id: object
   • title: object
   • description: object
   • price: float64
   • currency: object
   • category_id: object
   • category_name: object
   • brand_id: object
   • brand_name: object
   • seller_id: object
   • seller_name: object
   • seller_reputati

In [5]:
# Usando o DataFrame já carregado (df_products)
df = df_products.copy()

# Exploração detalhada da estrutura dos dados
print("🔍 EXPLORAÇÃO DETALHADA DOS DADOS")
print("=" * 50)

# 1. Verificar se temos dados válidos
if len(df) == 0:
    print("❌ ERRO: Dataset vazio!")
else:
    print(f"✅ Dataset carregado com {len(df)} registros")

# 2. Primeiros registros para verificar estrutura
print(f"\n📋 PRIMEIROS 3 REGISTROS:")
print(df.head(3))

# 3. Informações detalhadas sobre as colunas
print(f"\n📊 INFORMAÇÕES DAS COLUNAS:")
print(df.info())

🔍 EXPLORAÇÃO DETALHADA DOS DADOS
✅ Dataset carregado com 100 registros

📋 PRIMEIROS 3 REGISTROS:
                                     id                 title  \
0  501dcf86-801f-43fb-aa6f-af4b4f8ddb0d  Apple Gabinete Gamer   
1  4862fe26-1268-4359-be0e-e73138ef039d          Perfume Sony   
2  3e0b9e72-87e6-4640-9b27-1ab5c8b1692c  Samsung Jogo para PC   

                                         description   price currency  \
0  Sapiente ea a. Cum saepe libero a sit.\nDolore...  720.72      BRL   
1  Dolore tenetur labore eum facilis occaecati ac...  206.91      BRL   
2  Fugiat dicta repellat. Qui accusamus exercitat...  321.21      BRL   

  category_id        category_name brand_id brand_name seller_id  \
0       cat_2          Smartphones  brand_2      Apple  seller_8   
1      cat_10  Calçados Esportivos  brand_6       Sony  seller_4   
2       cat_2          Smartphones  brand_1    Samsung  seller_9   

        seller_name  seller_reputation  stock_quantity  views  sales  rating

## 4. Visualizações Básicas

Criamos gráficos básicos para entender a distribuição dos dados.

In [ ]:
# Configurar o tamanho das figuras
plt.rcParams['figure.figsize'] = (15, 10)

# Criar subplots para várias visualizações
fig, axes = plt.subplots(2, 3, figsize=(18, 12))
fig.suptitle('📊 Análise Exploratória de Produtos do Marketplace', fontsize=16, fontweight='bold')

# 1. Distribuição de Preços
axes[0, 0].hist(df_products['price'], bins=30, color='skyblue', alpha=0.7, edgecolor='black')
axes[0, 0].set_title('💰 Distribuição de Preços')
axes[0, 0].set_xlabel('Preço (R$)')
axes[0, 0].set_ylabel('Frequência')
axes[0, 0].grid(True, alpha=0.3)

# 2. Produtos por Categoria
category_counts = df_products['category_name'].value_counts()
axes[0, 1].bar(category_counts.index, category_counts.values, color='lightcoral')
axes[0, 1].set_title('🏷️ Produtos por Categoria')
axes[0, 1].set_xlabel('Categoria')
axes[0, 1].set_ylabel('Número de Produtos')
axes[0, 1].tick_params(axis='x', rotation=45)

# 3. Distribuição de Estoque
axes[0, 2].hist(df_products['stock_quantity'], bins=25, color='lightgreen', alpha=0.7, edgecolor='black')
axes[0, 2].set_title('📦 Distribuição de Estoque')
axes[0, 2].set_xlabel('Quantidade em Estoque')
axes[0, 2].set_ylabel('Frequência')
axes[0, 2].grid(True, alpha=0.3)

# 4. Produtos por Marca
brand_counts = df_products['brand_name'].value_counts()
axes[1, 0].bar(brand_counts.index, brand_counts.values, color='gold')
axes[1, 0].set_title('🏢 Produtos por Marca')
axes[1, 0].set_xlabel('Marca')
axes[1, 0].set_ylabel('Número de Produtos')
axes[1, 0].tick_params(axis='x', rotation=45)

# 5. Reputação dos Vendedores
axes[1, 1].hist(df_products['seller_reputation'], bins=20, color='mediumpurple', alpha=0.7, edgecolor='black')
axes[1, 1].set_title('⭐ Distribuição de Reputação dos Vendedores')
axes[1, 1].set_xlabel('Reputação (1-5)')
axes[1, 1].set_ylabel('Frequência')
axes[1, 1].grid(True, alpha=0.3)

# 6. Status dos Produtos (corrigido para usar 'status')
status_counts = df_products['status'].value_counts()
axes[1, 2].pie(status_counts.values, labels=status_counts.index, autopct='%1.1f%%', 
               colors=['lightgreen', 'lightcoral', 'lightyellow'], startangle=90)
axes[1, 2].set_title('🔄 Status dos Produtos')

plt.tight_layout()
plt.show()

# Estatísticas resumidas
print("\n📈 ESTATÍSTICAS PRINCIPAIS:")
print(f"💰 Preço médio: R$ {df_products['price'].mean():.2f}")
print(f"💰 Preço mediano: R$ {df_products['price'].median():.2f}")
print(f"📦 Estoque médio: {df_products['stock_quantity'].mean():.0f} unidades")
print(f"⭐ Reputação média: {df_products['seller_reputation'].mean():.2f}")
print(f"🏷️ Categorias únicas: {df_products['category_name'].nunique()}")
print(f"🏢 Marcas únicas: {df_products['brand_name'].nunique()}")
print(f"👤 Vendedores únicos: {df_products['seller_name'].nunique()}")
print(f"📊 Visualizações médias: {df_products['views'].mean():.0f}")
print(f"🛒 Vendas médias: {df_products['sales'].mean():.0f}")
print(f"⭐ Rating médio: {df_products['rating'].mean():.1f}")
print(f"\n🔄 Distribuição por Status:")
for status, count in status_counts.items():
    print(f"   • {status}: {count} produtos ({count/len(df_products)*100:.1f}%)")

## 5. Analytics Avançados

Análises mais sofisticadas para descobrir padrões e correlações nos dados.

In [ ]:
# 1. Matriz de Correlação
plt.figure(figsize=(12, 8))
numeric_columns = ['price', 'stock_quantity', 'seller_reputation']
correlation_matrix = df_products[numeric_columns].corr()

sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', center=0, 
            square=True, linewidths=0.5, cbar_kws={"shrink": .8})
plt.title('🔗 Matriz de Correlação entre Variáveis Numéricas', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

# 2. Boxplot de Preços por Categoria
plt.figure(figsize=(14, 8))
sns.boxplot(data=df_products, x='category_name', y='price')
plt.title('📊 Distribuição de Preços por Categoria', fontsize=14, fontweight='bold')
plt.xlabel('Categoria')
plt.ylabel('Preço (R$)')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

# 3. Relação entre Reputação do Vendedor e Preço
plt.figure(figsize=(12, 6))
plt.subplot(1, 2, 1)
plt.scatter(df_products['seller_reputation'], df_products['price'], alpha=0.6, color='blue')
plt.xlabel('Reputação do Vendedor')
plt.ylabel('Preço (R$)')
plt.title('💎 Reputação vs Preço')
plt.grid(True, alpha=0.3)

# 4. Preço médio por marca
plt.subplot(1, 2, 2)
brand_avg_price = df_products.groupby('brand_name')['price'].mean().sort_values(ascending=False)
plt.bar(brand_avg_price.index, brand_avg_price.values, color='orange', alpha=0.7)
plt.xlabel('Marca')
plt.ylabel('Preço Médio (R$)')
plt.title('🏢 Preço Médio por Marca')
plt.xticks(rotation=45)

plt.tight_layout()
plt.show()

# 5. Análise temporal (se dados de data estiverem disponíveis)
if 'created_at' in df_products.columns and df_products['created_at'].notna().any():
    plt.figure(figsize=(15, 6))
    
    # Produtos criados por mês
    df_products['month_year'] = df_products['created_at'].dt.to_period('M')
    monthly_products = df_products.groupby('month_year').size()
    
    plt.subplot(1, 2, 1)
    monthly_products.plot(kind='line', marker='o', color='green')
    plt.title('📈 Produtos Criados por Mês')
    plt.xlabel('Mês/Ano')
    plt.ylabel('Número de Produtos')
    plt.xticks(rotation=45)
    plt.grid(True, alpha=0.3)
    
    # Preço médio por mês
    monthly_avg_price = df_products.groupby('month_year')['price'].mean()
    plt.subplot(1, 2, 2)
    monthly_avg_price.plot(kind='line', marker='s', color='red')
    plt.title('💰 Preço Médio por Mês')
    plt.xlabel('Mês/Ano')
    plt.ylabel('Preço Médio (R$)')
    plt.xticks(rotation=45)
    plt.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()

# 6. Top 10 produtos mais caros
print("\n💎 TOP 10 PRODUTOS MAIS CAROS:")
top_expensive = df_products.nlargest(10, 'price')[['title', 'price', 'category_name', 'brand_name']]
display(top_expensive)

# 7. Análise por vendedor
print("\n👤 ANÁLISE POR VENDEDOR:")
seller_analysis = df_products.groupby('seller_name').agg({
    'price': ['mean', 'count'],
    'seller_reputation': 'first',
    'stock_quantity': 'sum'
}).round(2)

seller_analysis.columns = ['Preço Médio', 'Qtd Produtos', 'Reputação', 'Estoque Total']
seller_analysis = seller_analysis.sort_values('Qtd Produtos', ascending=False)
display(seller_analysis)

## 6. Filtros Interativos e Análises Personalizadas

Ferramentas para exploração dinâmica dos dados com filtros personalizáveis.

In [ ]:
def filter_products(df, 
                   min_price=None, max_price=None,
                   categories=None, brands=None, sellers=None,
                   min_stock=None, min_reputation=None,
                   active_only=False):
    """
    Filtra produtos baseado em critérios especificados
    
    Parâmetros:
    - min_price, max_price: filtros de preço
    - categories, brands, sellers: listas de valores para filtrar
    - min_stock: estoque mínimo
    - min_reputation: reputação mínima do vendedor
    - active_only: mostrar apenas produtos ativos
    """
    filtered_df = df.copy()
    
    if active_only:
        filtered_df = filtered_df[filtered_df['active'] == True]
    
    if min_price is not None:
        filtered_df = filtered_df[filtered_df['price'] >= min_price]
    
    if max_price is not None:
        filtered_df = filtered_df[filtered_df['price'] <= max_price]
    
    if categories:
        filtered_df = filtered_df[filtered_df['category_name'].isin(categories)]
    
    if brands:
        filtered_df = filtered_df[filtered_df['brand_name'].isin(brands)]
    
    if sellers:
        filtered_df = filtered_df[filtered_df['seller_name'].isin(sellers)]
    
    if min_stock is not None:
        filtered_df = filtered_df[filtered_df['stock_quantity'] >= min_stock]
    
    if min_reputation is not None:
        filtered_df = filtered_df[filtered_df['seller_reputation'] >= min_reputation]
    
    return filtered_df

def analyze_filtered_data(filtered_df, title="Análise dos Dados Filtrados"):
    """
    Realiza análise rápida dos dados filtrados
    """
    print(f"\n🔍 {title.upper()}")
    print("=" * 60)
    print(f"📊 Total de produtos: {len(filtered_df)}")
    
    if len(filtered_df) == 0:
        print("❌ Nenhum produto encontrado com os filtros aplicados.")
        return
    
    print(f"💰 Preço médio: R$ {filtered_df['price'].mean():.2f}")
    print(f"💰 Faixa de preço: R$ {filtered_df['price'].min():.2f} - R$ {filtered_df['price'].max():.2f}")
    print(f"📦 Estoque total: {filtered_df['stock_quantity'].sum():.0f} unidades")
    print(f"⭐ Reputação média: {filtered_df['seller_reputation'].mean():.2f}")
    
    print(f"\n🏷️ Top 3 categorias:")
    top_categories = filtered_df['category_name'].value_counts().head(3)
    for category, count in top_categories.items():
        print(f"   • {category}: {count} produtos")
    
    print(f"\n🏢 Top 3 marcas:")
    top_brands = filtered_df['brand_name'].value_counts().head(3)
    for brand, count in top_brands.items():
        print(f"   • {brand}: {count} produtos")
    
    return filtered_df

# Exemplos de uso dos filtros
print("🔍 EXEMPLOS DE FILTROS PERSONALIZADOS")
print("=" * 50)

# Exemplo 1: Produtos de luxo (preço > R$ 500)
luxury_products = filter_products(df_products, min_price=500)
analyze_filtered_data(luxury_products, "Produtos de Luxo (> R$ 500)")

# Exemplo 2: Produtos em estoque com boa reputação
quality_products = filter_products(df_products, min_stock=10, min_reputation=4.0)
analyze_filtered_data(quality_products, "Produtos com Estoque e Boa Reputação")

# Exemplo 3: Produtos específicos por categoria
if 'category_name' in df_products.columns:
    available_categories = df_products['category_name'].unique()
    if len(available_categories) > 0:
        sample_category = [available_categories[0]]
        category_products = filter_products(df_products, categories=sample_category)
        analyze_filtered_data(category_products, f"Produtos da Categoria: {sample_category[0]}")

print("\n" + "="*60)
print("💡 DICAS DE USO:")
print("• Modifique os parâmetros da função filter_products() para criar seus próprios filtros")
print("• Use analyze_filtered_data() para obter insights rápidos dos dados filtrados")
print("• Combine múltiplos filtros para análises mais específicas")
print("• Execute as células acima novamente após modificar os filtros")

## 7. Gráficos Interativos com Plotly

Visualizações interativas para exploração mais dinâmica dos dados.

In [ ]:
# 1. Gráfico de dispersão interativo: Preço vs Estoque
fig1 = px.scatter(df_products, 
                  x='stock_quantity', 
                  y='price',
                  color='category_name',
                  size='seller_reputation',
                  hover_data=['title', 'brand_name', 'seller_name'],
                  title='📊 Preço vs Estoque (Tamanho = Reputação do Vendedor)',
                  labels={'stock_quantity': 'Quantidade em Estoque',
                          'price': 'Preço (R$)',
                          'category_name': 'Categoria'})

fig1.update_layout(height=600)
fig1.show()

# 2. Gráfico de barras interativo: Produtos por categoria
category_counts = df_products['category_name'].value_counts()
fig2 = px.bar(x=category_counts.index, 
              y=category_counts.values,
              title='🏷️ Distribuição de Produtos por Categoria',
              labels={'x': 'Categoria', 'y': 'Número de Produtos'},
              color=category_counts.values,
              color_continuous_scale='viridis')

fig2.update_layout(height=500)
fig2.show()

# 3. Boxplot interativo: Preços por marca
fig3 = px.box(df_products, 
              x='brand_name', 
              y='price',
              title='📦 Distribuição de Preços por Marca',
              labels={'brand_name': 'Marca', 'price': 'Preço (R$)'})

fig3.update_layout(height=500)
fig3.update_xaxes(tickangle=45)
fig3.show()

# 4. Histograma interativo: Distribuição de reputação
fig4 = px.histogram(df_products, 
                    x='seller_reputation',
                    nbins=20,
                    title='⭐ Distribuição de Reputação dos Vendedores',
                    labels={'seller_reputation': 'Reputação do Vendedor',
                            'count': 'Frequência'})

fig4.update_layout(height=400)
fig4.show()

# 5. Mapa de calor de vendedores vs categorias
if len(df_products) > 0:
    seller_category_matrix = pd.crosstab(df_products['seller_name'], df_products['category_name'])
    
    fig5 = px.imshow(seller_category_matrix.values,
                     x=seller_category_matrix.columns,
                     y=seller_category_matrix.index,
                     color_continuous_scale='Blues',
                     title='🔥 Mapa de Calor: Vendedores vs Categorias',
                     labels={'x': 'Categoria', 'y': 'Vendedor', 'color': 'Número de Produtos'})
    
    fig5.update_layout(height=600)
    fig5.show()

print("✨ Gráficos interativos criados com sucesso!")
print("💡 Dica: Passe o mouse sobre os pontos para ver mais informações")
print("🔍 Use as ferramentas de zoom e seleção para explorar os dados")

## 8. Exportar Dados e Conclusões

Ferramentas para exportar dados filtrados e gerar relatórios.

In [ ]:
def export_analysis_report(df, filename='product_analysis_report'):
    """
    Exporta um relatório completo da análise em diferentes formatos
    """
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    
    # 1. Exportar dados em CSV
    csv_filename = f"{filename}_{timestamp}.csv"
    df.to_csv(csv_filename, index=False, encoding='utf-8')
    print(f"📄 Dados exportados para: {csv_filename}")
    
    # 2. Criar relatório estatístico
    report_filename = f"{filename}_summary_{timestamp}.txt"
    with open(report_filename, 'w', encoding='utf-8') as f:
        f.write("=" * 60 + "\n")
        f.write("RELATÓRIO DE ANÁLISE DE PRODUTOS - MARKETPLACE\n")
        f.write("=" * 60 + "\n\n")
        
        f.write(f"Data/Hora da Análise: {datetime.now().strftime('%d/%m/%Y %H:%M:%S')}\n")
        f.write(f"Total de Produtos Analisados: {len(df)}\n\n")
        
        f.write("ESTATÍSTICAS PRINCIPAIS:\n")
        f.write("-" * 30 + "\n")
        if 'price' in df.columns:
            f.write(f"Preço Médio: R$ {df['price'].mean():.2f}\n")
            f.write(f"Preço Mediano: R$ {df['price'].median():.2f}\n")
            f.write(f"Faixa de Preços: R$ {df['price'].min():.2f} - R$ {df['price'].max():.2f}\n")
        
        if 'stock_quantity' in df.columns:
            f.write(f"Estoque Total: {df['stock_quantity'].sum():.0f} unidades\n")
            f.write(f"Estoque Médio por Produto: {df['stock_quantity'].mean():.1f} unidades\n")
        
        if 'seller_reputation' in df.columns:
            f.write(f"Reputação Média dos Vendedores: {df['seller_reputation'].mean():.2f}\n")
        
        f.write(f"\nCategorias Únicas: {df['category_name'].nunique()}\n")
        f.write(f"Marcas Únicas: {df['brand_name'].nunique()}\n")
        f.write(f"Vendedores Únicos: {df['seller_name'].nunique()}\n\n")
        
        f.write("TOP 5 CATEGORIAS:\n")
        f.write("-" * 20 + "\n")
        top_categories = df['category_name'].value_counts().head(5)
        for i, (category, count) in enumerate(top_categories.items(), 1):
            f.write(f"{i}. {category}: {count} produtos\n")
        
        f.write("\nTOP 5 MARCAS:\n")
        f.write("-" * 15 + "\n")
        top_brands = df['brand_name'].value_counts().head(5)
        for i, (brand, count) in enumerate(top_brands.items(), 1):
            f.write(f"{i}. {brand}: {count} produtos\n")
    
    print(f"📊 Relatório estatístico salvo em: {report_filename}")
    
    # 3. Exportar dados em Excel (se xlsxwriter estiver disponível)
    try:
        excel_filename = f"{filename}_{timestamp}.xlsx"
        with pd.ExcelWriter(excel_filename, engine='xlsxwriter') as writer:
            # Dados completos
            df.to_excel(writer, sheet_name='Dados_Completos', index=False)
            
            # Resumo por categoria
            category_summary = df.groupby('category_name').agg({
                'price': ['mean', 'count', 'min', 'max'],
                'stock_quantity': 'sum',
                'seller_reputation': 'mean'
            }).round(2)
            category_summary.to_excel(writer, sheet_name='Resumo_Categorias')
            
            # Resumo por marca
            brand_summary = df.groupby('brand_name').agg({
                'price': ['mean', 'count'],
                'stock_quantity': 'sum'
            }).round(2)
            brand_summary.to_excel(writer, sheet_name='Resumo_Marcas')
        
        print(f"📈 Planilha Excel criada: {excel_filename}")
    except ImportError:
        print("⚠️  xlsxwriter não está instalado. Instale com: pip install xlsxwriter")
    
    return csv_filename, report_filename

# Função para criar filtros personalizados e exportar
def create_custom_filter():
    """
    Interface simples para criar filtros personalizados
    """
    print("\n🔧 CRIADOR DE FILTROS PERSONALIZADOS")
    print("=" * 40)
    print("Modifique os valores abaixo para criar seu filtro:")
    print("\n# Exemplo de uso:")
    print("filtered_data = filter_products(")
    print("    df_products,")
    print("    min_price=50,           # Preço mínimo")
    print("    max_price=500,          # Preço máximo") 
    print("    categories=['Eletrônicos', 'Roupas'],  # Categorias específicas")
    print("    min_stock=5,            # Estoque mínimo")
    print("    min_reputation=3.0      # Reputação mínima")
    print(")")
    print("\n# Para exportar os dados filtrados:")
    print("export_analysis_report(filtered_data, 'meu_filtro_personalizado')")
    
    print(f"\n📋 VALORES DISPONÍVEIS NO DATASET:")
    print(f"Categorias: {list(df_products['category_name'].unique())}")
    print(f"Marcas: {list(df_products['brand_name'].unique())}")
    print(f"Vendedores: {list(df_products['seller_name'].unique())}")
    print(f"Faixa de preços: R$ {df_products['price'].min():.2f} - R$ {df_products['price'].max():.2f}")

# Executar exemplos
print("💾 EXPORTANDO ANÁLISE COMPLETA DOS DADOS...")
export_analysis_report(df_products, 'marketplace_analysis_complete')

print("\n" + "="*60)
create_custom_filter()

print("\n" + "="*60)
print("🎉 NOTEBOOK DE ANÁLISE CONCLUÍDO!")
print("✅ Você agora tem ferramentas completas para:")
print("   • Carregar dados do Elasticsearch")
print("   • Explorar e visualizar informações de produtos")
print("   • Criar filtros personalizados") 
print("   • Gerar gráficos interativos")
print("   • Exportar relatórios e dados")
print("\n💡 Dicas finais:")
print("   • Execute as células em ordem para melhor experiência")
print("   • Modifique os filtros para suas necessidades específicas")
print("   • Use os gráficos interativos para explorar padrões")
print("   • Exporte os dados filtrados para análises externas")